# Parameter Study / Control Panel (Pencil Code)
- Create/clone sims
- Sweep & change parameters from a single dict/DataFrame
- Run them (optional)
- Read & plot time series (μ̃5, S5) with vertical markers
- Read & plot spectra (mag, hel_mag, GWs) and mark k_cpi



- 5001: change of source and eta
- 5002: change of t_phi
- 5003: bigger grid 
- 5004: decres od gamma

In [1]:
# %% Imports
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl



import pencil as pc
from pencil import read, io

plt.rcParams["figure.dpi"] = 130
plt.rcParams["text.usetex"] = True  # if you want TeX


$ conda install -c plotly plotly-orca psutil requests


In [2]:
# %% Paths
TEMP = "5001"
SIM_ROOT   = os.path.expanduser("~/programming/test/mhd_project")
PROJECTDIR = os.path.join(SIM_ROOT, "tmp", TEMP)   # where clones go
FIG_DIR    = os.path.join(SIM_ROOT, "figs", TEMP)
os.makedirs(FIG_DIR, exist_ok=True)

# Base simulation you copy from:
BASE_SIM_PATH = os.path.join(SIM_ROOT, "runs_directory2", "smooth_source2_mu0")  # change as needed
SIM_BASE = pc.get_sim(BASE_SIM_PATH)
print("Base sim:", SIM_BASE.name)

Base sim: smooth_source2_mu0


In [3]:
# %% Parameter sweep definition
# row = one simulation. Key = sim name.
param_df = pd.DataFrame([
    #   name   source5  source5_expt2  gammaf5  eta nt   
    ("AB0",   1e10,    0.05,          1e3,     1e-8,800),
    ("AB1",   1e10,    0.1,          1e3,     1e-8,800),
    ("AB2",   1e10,    0.2,          1e3,     1e-8,800),
    ("AB3",   1e10,    0.4,          1e3,     1e-8,800),
], columns=["name", "source5", "source5_expt2", "gammaf5", "eta","nt"]).set_index("name")

param_df


,source5,source5_expt2,gammaf5,eta,nt
name,,,,,
AB0,1.000000e+10,0.05,1000.0,1.000000e-08,800
AB1,1.000000e+10,0.10,1000.0,1.000000e-08,800
AB2,1.000000e+10,0.20,1000.0,1.000000e-08,800
AB3,1.000000e+10,0.40,1000.0,1.000000e-08,800


In [4]:
# %% Save parameter sweep to CSV
import os
import pandas as pd

SWEEP_DIR = os.path.join(SIM_ROOT, "param_sweeps",TEMP)
os.makedirs(SWEEP_DIR, exist_ok=True)

STAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = os.path.join(SWEEP_DIR, f"sweep_{STAMP}.csv")

param_df.to_csv(CSV_PATH)      # index ('name') is kept as first column
print("Saved sweep to:", CSV_PATH)


Saved sweep to: /home/mgurgeni/programming/test/mhd_project/param_sweeps/5001/sweep_20250728_122152.csv


In [5]:
# %% Load a parameter sweep from CSV
def load_sweep(csv_path=None, sweep_dir=SWEEP_DIR):
    """
    Load a sweep CSV. If csv_path is None, pick the newest file in sweep_dir.
    Returns a DataFrame indexed by 'name'.
    """
    if csv_path is None:
        files = sorted(
            [f for f in os.listdir(sweep_dir) if f.endswith(".csv")]
        )
        if not files:
            raise FileNotFoundError("No CSV sweeps found in " + sweep_dir)
        csv_path = os.path.join(sweep_dir, files[-1])
    df = pd.read_csv(csv_path).set_index("name")
    print("Loaded sweep from:", csv_path)
    return df

# Example usage:
param_df = load_sweep()                          # latest
# param_df = load_sweep("param_sweeps/sweep_....csv")  # specific file


Loaded sweep from: /home/mgurgeni/programming/test/mhd_project/param_sweeps/5001/sweep_20250728_122152.csv


In [6]:
# %% Create sims & apply params (fixed)
SIMS = []

for sim_name, row in param_df.iterrows():
    dst = os.path.join(PROJECTDIR, sim_name)

    # create or fetch
    if not os.path.isdir(dst):
        SIM = SIM_BASE.copy(path_root=PROJECTDIR, name=sim_name)
    else:
        SIM = pc.get_sim(dst, quiet=True)

    # change parameters using Pencil helpers
    for q in ["source5", "source5_expt2", "gammaf5", "eta","nt"]:
        io.change_value_in_file(
            filename="run.in",
            quantity=q,
            newValue=row[q],
            sim=SIM,                # <- key fix (or filepath=SIM.datadir)
            DEBUG=False
        )

    SIMS.append(SIM)

print(len(SIMS), [s.name for s in SIMS])


4 ['AB0', 'AB1', 'AB2', 'AB3']


In [7]:
# %% Time series helpers
def read_ts(sim):
    return read.ts(datadir=sim.datadir, file_name="time_series.dat")

def plot_mu_S5(ts, params, ax=None, label=None, vlines=None, save=None):
    if ax is None:
        fig, ax = plt.subplots()
    t   = ts.t - 1
    mu5 = ts.mu5m
    S5  = ts.srce5m

    ax.loglog(t, mu5, "-x", label=(label or r"$\tilde{\mu}_5$"))
    ax.loglog(t, S5/params.gammaf5, "--", label=r"$S_5/\Gamma$")
    for x, text in vlines:
        ax.axvline(x, ls="--", lw=0.8, color="grey")
        ax.text(x*1.03, ax.get_ylim()[1]/5, text, rotation=90, va="top", fontsize=9)
    ax.set_xlabel(r"$t$")
    ax.set_ylabel(r"values")
    ax.grid(alpha=0.3)
    ax.legend()
    if save:
        plt.savefig(save, bbox_inches="tight")
        plt.close()

# Example vertical markers
def compute_markers(p):
    t_gamma = 1/p.gammaf5
    t_phi   = p.source5_expt2
    
    t_cross = 2**(2/3) * t_phi ** (2/3) * t_gamma ** (-2/3) * p.source5 ** (-2/3) * p.eta ** (1/3)
    return [(t_gamma, r"$t=1/\Gamma$"), (t_phi, r"$t=t_\phi$"), (t_cross, r"$t_{C}$")]


In [8]:
# %% Spectra helpers
def read_power(sim):
    return read.power(datadir=sim.datadir)

def get_kcpi(ts, par):
    val = np.asarray(ts.mu5m)[0]
    return np.abs(val) if np.isfinite(val) else None
  
def plot_spec(ps, key, kcpi=None, label=None, save=None):
    fig, ax = plt.subplots()

    arr = getattr(ps, key)
    k   = ps.krms
    Pk  = arr[-1] if arr.ndim == 2 else arr

    ax.loglog(k, np.abs(Pk), label=label or key)

    # guard kcpi
    if (kcpi is not None and np.isfinite(kcpi)
        and k.min() < kcpi < k.max()):
        ax.axvline(kcpi, ls="--", lw=0.8)
        ymax = np.nanmax(Pk)
        ax.text(kcpi*1.05, ymax*0.7, r"$k_{\rm cpi}$",
                rotation=90, fontsize=9, clip_on=True)

    ax.set_xlabel(r"$k$")
    ax.set_ylabel(r"$P(k)$")
    ax.grid(alpha=0.3)
    ax.legend()

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=200)
        plt.close(fig)            # <-- important
        return                    # <-- nothing returned, nothing rendered
    else:
        plt.show()

In [9]:
for SIM in SIMS:
    name = SIM.name
    try:
        ts  = read_ts(SIM)
        par = read.param(datadir=SIM.datadir, param2=True)
        vl  = compute_markers(par)

        plot_mu_S5(ts, par, vlines=vl,
                   save=os.path.join(FIG_DIR, f"{name}_ts.png"),
                   label=name)

    except Exception as e:
        print(f"{name} -> {e}")



Read 400 lines.
Read 400 lines.
Read 400 lines.
Read 400 lines.


In [10]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_spectra_vs_k_all_times(sim, field="mag", label="", out=None,
                                log_t=False, t_offset=1.0, cmap="viridis"):
    """
    Plot P(k,t) for all time snapshots as log-log lines.
    Each line gets a color from a colormap keyed to time; a colorbar is added.

    Parameters
    ----------
    sim_power : object
        Result of pc.read.power(datadir=...).
    field : str
        Attribute in sim_power (e.g. "mag", "hel_mag", "GWs").
    label : str
        Title/legend label for the figure.
    out : str or None
        Path to save figure (PNG). If None -> show().
    log_t : bool
        Use log10(t) for color scale.
    t_offset : float
        Subtract this from times (your data seems to use t-1).
    cmap : str
        Matplotlib colormap name.
    """
    sim_power = read_power(sim)
    ts = read_ts(sim)
    par = read.param(datadir=SIM.datadir, param2=True)
    k   = sim_power.krms
    arr = getattr(sim_power, field)   # shape (nt, nk)
    t   = np.asarray(sim_power.t) - t_offset
    kcpi = get_kcpi(ts,par)

    # color mapping
    times_for_cmap = np.log10(t) if log_t else t
    norm = mpl.colors.Normalize(vmin=times_for_cmap.min(), vmax=times_for_cmap.max())
    sm   = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)

    fig, ax = plt.subplots()

    for i in range(arr.shape[0]):
        c = sm.to_rgba(times_for_cmap[i])
        ax.loglog(k*1000, np.abs(arr[i]), color=c, lw=1.0) ## 1000 is wav1 in run.in

    ax.set_xlabel(r"$k$")
    ax.set_ylabel(r"$P(k)$")
    ttl = f"{field} spectrum {label}" if label else f"{field} spectrum"
    ax.set_title(ttl)
    ax.grid(alpha=0.3)

    # colorbar
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label(r"$\log_{10}(t)$" if log_t else r"$t$")

    if out:
        fig.savefig(out, dpi=220, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


In [11]:
for sim in SIMS:
    for fld in ["mag", "hel_mag", "GWs"]:
        if hasattr(pc.read.power(datadir=sim.datadir), fld):
            out = os.path.join(FIG_DIR, f"{sim.name}_{fld}_alltimes.png")
            plot_spectra_vs_k_all_times(sim, field=fld, label=sim.name, out=out,
                                        log_t=False, t_offset=1.0, cmap="plasma")
        else:
            print(f"{sim.name}: no {fld}")



power_GWh.dat
powerhel_GWs.dat
power_Tpq.dat
power_Str.dat
powerhel_kin.dat
power_GWs.dat
powerhel_mag.dat
power_sp.dat
power_krms.dat
power_SCL.dat
power_mag.dat
powerhel_Str.dat
power_VCT.dat
power_kin.dat
powerhel_GWh.dat
power_GWh.dat
powerhel_GWs.dat
power_Tpq.dat
power_Str.dat
powerhel_kin.dat
power_GWs.dat
powerhel_mag.dat
power_sp.dat
power_krms.dat
power_SCL.dat
power_mag.dat
powerhel_Str.dat
power_VCT.dat
power_kin.dat
powerhel_GWh.dat
Read 400 lines.
power_GWh.dat
powerhel_GWs.dat
power_Tpq.dat
power_Str.dat
powerhel_kin.dat
power_GWs.dat
powerhel_mag.dat
power_sp.dat
power_krms.dat
power_SCL.dat
power_mag.dat
powerhel_Str.dat
power_VCT.dat
power_kin.dat
powerhel_GWh.dat
power_GWh.dat
powerhel_GWs.dat
power_Tpq.dat
power_Str.dat
powerhel_kin.dat
power_GWs.dat
powerhel_mag.dat
power_sp.dat
power_krms.dat
power_SCL.dat
power_mag.dat
powerhel_Str.dat
power_VCT.dat
power_kin.dat
powerhel_GWh.dat
Read 400 lines.
power_GWh.dat
powerhel_GWs.dat
power_Tpq.dat
power_Str.dat
powerhel_

In [12]:
# %% Record parameters actually applied
rows=[]
for SIM in SIMS:
    p = read.param(datadir=SIM.datadir, param2=True)
    rows.append(dict(run=SIM.name,
                     source5=p.source5,
                     tphi=p.source5_expt2,
                     gammaf5=p.gammaf5,
                     eta=p.eta,
                     Diffmu5=p.diffmu5))
pd.DataFrame(rows).set_index("run")


,source5,tphi,gammaf5,eta,Diffmu5
run,,,,,
AB0,100000000.0,0.05,1000.0,1.000000e-06,1.000000e-06
AB1,100000000.0,0.05,1000.0,1.000000e-06,1.000000e-06
AB2,100000000.0,0.05,1000.0,1.000000e-08,1.000000e-06
AB3,100000000.0,0.05,1000.0,1.000000e-06,1.000000e-06
